<a href="https://colab.research.google.com/github/larryjay007/MyML/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
%pip -q install duckdb huggingface_hub

One row = one content item's monthly aggregate: a single page's (content_hash_id) total
search performance for one calendar month, built from fact_content_daily_performance.

Time window: features are built from March 2026 (month=2026-03) — a mid-panel month, per
the assignment's warning that the final month (June 2026, the _sample table) is a sealed
test month and must never be used to develop label logic. The label is built from April
2026 (month=2026-04) — the month immediately after the feature window, so the label
represents a genuinely future, observed outcome rather than something computed from the
same window as the features.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [ ]:
label = con.sql(f"""
    SELECT content_hash_id, client_hash_id, SUM(gsc_impressions) AS impressions_apr
    FROM {APRIL}
    GROUP BY content_hash_id, client_hash_id
""").df()

data = features.merge(label, on=['content_hash_id', 'client_hash_id'], how='inner')
data['is_declining'] = (data['impressions_apr'] < 0.8 * data['impressions_mar']).astype(int)

print(f"{len(data):,} pages with both March features and an April outcome")
print(f"Declining rate: {data['is_declining'].mean():.1%}")
data[['content_hash_id', 'impressions_mar', 'impressions_apr', 'is_declining']].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

143,206 pages with both March features and an April outcome
Declining rate: 51.7%


,content_hash_id,impressions_mar,impressions_apr,is_declining
0,content_b7e512995f79d5a6,1140.0,1151.0,0
1,content_a7da352b73b02668,4944.0,6091.0,0
2,content_d056587ff7faca0c,2770.0,6266.0,0
3,content_bfd1e41c2af250c8,48.0,98.0,0
4,content_2662845f598544ef,150.0,100.0,1


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ['impressions_mar', 'clicks_mar', 'avg_position_mar', 'ctr_mar', 'active_days_mar']
model_data = data.dropna(subset=honest_features)

X, y = model_data[honest_features], model_data['is_declining']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
honest_score = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

print(f"Honest baseline ROC-AUC (5 real features): {honest_score:.3f}")

Honest baseline ROC-AUC (5 real features): 0.607


The trap: including impressions_apr — the exact column the label is derived from — as a
"feature" pushed ROC-AUC from 0.607 to 0.999. That's not a better model; it's a model that
can see the answer. is_declining is literally computed by comparing impressions_apr to
impressions_mar, so handing the model impressions_apr directly is close to handing it the
label itself.

The leaky column and leaky model are discarded. The honest number — 0.607, using only the
5 features knowable at the end of March — is the real result. It's a modest score, not an
exciting one, and that's the honest state of this slice: a real but limited signal, not
proof of a strong model yet.

FEATURES (knowable at the decision moment — end of March):
- impressions_mar — total March impressions for the page
- clicks_mar — total March clicks for the page
- avg_position_mar — average search position, March only
- ctr_mar — clicks_mar / impressions_mar
- active_days_mar — number of distinct days in March with any recorded impressions

LABEL / PROXY:
- is_declining — 1 if April impressions < 80% of March impressions, else 0. Built entirely
  from April data, which is never joined into the feature set. This is an observed future
  outcome (next month's real result), not a rule-computed bucket like the starter dataset's
  trend_direction proxy.

CONTEXT (grouping/joining only, never features):
- content_hash_id — identifies the page; used to group and join, carries no signal itself
- client_hash_id — identifies the client; used for client-grouped validation later, never
  fed to a model

EXCLUDED:
- April's row-level performance data (beyond what's needed to compute is_declining) —
  excluded because including it as a feature would mean the model sees the future it's
  supposed to predict. This is the exact rule Part 4's deliberate-leak experiment will
  intentionally break, to show what happens when it's violated.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# Scoped to just the two months we actually need — this keeps every query fast,
# since DuckDB only reads these partitions, never the full 79M-row table.
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

In [ ]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {MARCH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()

print(f"Duplicate grain rows found: {len(grain_check)}")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0


,report_date,client_hash_id,content_hash_id,c


In [ ]:
counts_check = con.sql(f"""
    SELECT COUNT(*) AS total_rows, MIN(report_date) AS earliest, MAX(report_date) AS latest,
           COUNT(DISTINCT content_hash_id) AS distinct_pages,
           COUNT(DISTINCT client_hash_id) AS distinct_clients
    FROM {MARCH}
""").df()

counts_check

,total_rows,earliest,latest,distinct_pages,distinct_clients
0,9841378,2026-03-01,2026-03-31,331437,55


In [ ]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_available
    FROM {MARCH}
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,9841378,413966.0,4.2


In [ ]:
features = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS impressions_mar,
        SUM(gsc_clicks) AS clicks_mar,
        AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_mar,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr_mar,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days_mar
    FROM {MARCH}
    GROUP BY content_hash_id, client_hash_id
    HAVING impressions_mar >= 10
""").df()

print(f"{len(features):,} pages with at least minimal March activity")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

143,206 pages with at least minimal March activity


,content_hash_id,client_hash_id,impressions_mar,clicks_mar,avg_position_mar,ctr_mar,active_days_mar
0,content_05597932fe4da067,client_73cda7b4e4f265ea,57.0,0.0,7.842593,0.000000,26
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,6523.0,7.0,7.209549,0.001073,31
2,content_905aa32a0230694e,client_73cda7b4e4f265ea,149.0,0.0,8.454069,0.000000,30
3,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,453.0,0.0,3.307255,0.000000,31
4,content_36c36abc7650d7af,client_73cda7b4e4f265ea,5630.0,6.0,6.724039,0.001066,31


Five features, all built from March only:

1. impressions_mar — knowable at the decision moment because it's a completed total for a
   month that has already fully passed by the time we'd act.
2. clicks_mar — same: a completed March total, no future data involved.
3. avg_position_mar — March's average position, computed only from March's own rows
   (avg_position = 0 excluded first, since it means "no data," not rank zero).
4. ctr_mar — derived purely from clicks_mar and impressions_mar, both already knowable.
5. active_days_mar — count of March days with any impressions; a completed historical count,
   not a forward-looking one.

All five are computed with a hard cutoff at March 31 — none touch April data in any way.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named limitation: this slice can only support GSC-based features (impressions, clicks,
position, CTR, active days) — not engagement-based ones. Query 3 showed only 4.2% of March
rows have GA4 data available, since most clients in this snapshot are GSC-only or hadn't
started GA4 tracking yet as of March. Any future feature relying on engagement_rate,
scroll_rate, or session data would only be usable for a small, non-representative slice of
pages, and building it without explicitly filtering on ga4_data_available IS TRUE would
risk treating "not tracked" as "zero engagement" — the exact trap the data skill warns
about by name.

A second, smaller limitation: the declining-rate label (51.7%) is based on a single
month-over-month comparison (March → April) for one snapshot in time. It doesn't capture
seasonality, and a page dropping from March to April could reflect a calendar effect
specific to those two months, not a general pattern that would hold for any other
month-pair.